In [7]:
import pandas as pd
import duckdb
from pathlib import Path
from IPython.display import display

def fix_missing_hr(df):
    """
    Fix missing HR values by linear interpolation between known values.
    
    For sequential null HR values, interpolates linearly between the last known
    HR value before the nulls and the first known HR value after the nulls.
    
    :param df: DataFrame with 'HR (bpm)' and 'time' columns
    :return: DataFrame with interpolated HR values
    """
    
    hr_key = 'HR (bpm)'

    # Make a copy to avoid modifying the original
    df_fixed = df.copy()
    
    # Find all null positions
    null_mask = df_fixed[hr_key].isnull()
    
    if not null_mask.any():
        print("No missing HR values found.")
        return df_fixed
    
    # Get groups of consecutive nulls
    null_groups = []
    in_null_group = False
    start_idx = None
    
    for i, is_null in enumerate(null_mask):
        if is_null and not in_null_group:
            # Start of a new null group
            start_idx = i
            in_null_group = True
        elif not is_null and in_null_group:
            # End of current null group
            null_groups.append((start_idx, i - 1))
            in_null_group = False
    
    # Handle case where nulls go to the end
    if in_null_group:
        null_groups.append((start_idx, len(df_fixed) - 1))
    
    print(f"Found {len(null_groups)} groups of consecutive null HR values")
    
    # Process each group of nulls
    for group_start, group_end in null_groups:
        null_count = group_end - group_start + 1
        
        # Find the last known HR value before nulls
        before_hr = None
        if group_start > 0:
            before_hr = df_fixed.iloc[group_start - 1][hr_key]

        # Find the first known HR value after nulls
        after_hr = None
        if group_end < len(df_fixed) - 1:
            after_hr = df_fixed.iloc[group_end + 1][hr_key]
        
        print(f"Null group: indices {group_start}-{group_end} ({null_count} nulls)")
        print(f"  Before HR: {before_hr}, After HR: {after_hr}")
        
        # Interpolate values
        if before_hr is not None and after_hr is not None:
            # Linear interpolation between two known values
            hr_diff = after_hr - before_hr
            delta_per_step = hr_diff / (null_count + 1)
            
            print(f"  HR difference: {hr_diff}, Delta per step: {delta_per_step:.2f}")
            
            for i, null_idx in enumerate(range(group_start, group_end + 1)):
                interpolated_value = before_hr + (i + 1) * delta_per_step
                df_fixed.iloc[null_idx, df_fixed.columns.get_loc(hr_key)] = interpolated_value

        elif before_hr is not None:
            # Only have before value - forward fill
            print(f"  Forward filling with HR: {before_hr}")
            for null_idx in range(group_start, group_end + 1):
                df_fixed.iloc[null_idx, df_fixed.columns.get_loc(hr_key)] = before_hr

        elif after_hr is not None:
            # Only have after value - backward fill
            print(f"  Backward filling with HR: {after_hr}")
            for null_idx in range(group_start, group_end + 1):
                df_fixed.iloc[null_idx, df_fixed.columns.get_loc(hr_key)] = after_hr
        else:
            print(f"  Warning: No surrounding HR values found for interpolation")
    
    # Summary
    remaining_nulls = df_fixed[hr_key].isnull().sum()
    fixed_nulls = null_mask.sum() - remaining_nulls
    
    print(f"\nInterpolation complete:")
    print(f"  Original null values: {null_mask.sum()}")
    print(f"  Fixed values: {fixed_nulls}")
    print(f"  Remaining nulls: {remaining_nulls}")
    
    return df_fixed

def import_workout_csv(csv_path: str, con: duckdb.DuckDBPyConnection):
    """
    Import workout CSV file into DuckDB database using the provided connection.
    
    :param csv_path: Path to the CSV file to import.
    :param con: DuckDB connection object.
    """
    
    try:
        # ------------------------------------------------------------------
        # 1.  Read the CSV and build the workoutId
        # ------------------------------------------------------------------
        print(f"Importing csv file '{csv_path}'")
        csv_file = Path(csv_path)
        
        # -- first row holds file-wide metadata
        df = pd.read_csv(csv_path, skiprows=2, nrows=200)
        print(df)
        df_fixed = fix_missing_hr(df)
        display(df_fixed)
    except Exception as e:
        print("❌ Error while importing workout CSV:")
        print(e)

import_workout_csv('./hr_data/Anton_Antonov+_2025-05-23_18-39-04.CSV', None)

Importing csv file './hr_data/Anton_Antonov+_2025-05-23_18-39-04.CSV'
     Sample rate      Time  HR (bpm)  Speed (km/h)  Pace (min/km)  Cadence  \
0            1.0  00:00:00       NaN           NaN            NaN      NaN   
1            NaN  00:00:01       NaN           NaN            NaN      NaN   
2            NaN  00:00:02       NaN           NaN            NaN      NaN   
3            NaN  00:00:03       NaN           NaN            NaN      NaN   
4            NaN  00:00:04       NaN           NaN            NaN      NaN   
..           ...       ...       ...           ...            ...      ...   
195          NaN  00:03:15     164.0           NaN            NaN      NaN   
196          NaN  00:03:16     165.0           NaN            NaN      NaN   
197          NaN  00:03:17     165.0           NaN            NaN      NaN   
198          NaN  00:03:18     166.0           NaN            NaN      NaN   
199          NaN  00:03:19     166.0           NaN            NaN      N

,Sample rate,Time,HR (bpm),Speed (km/h),Pace (min/km),Cadence,Altitude (m),Stride length (m),Distances (m),Temperatures (C),Power (W),Unnamed: 11
0,1.0,00:00:00,122.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,00:00:01,122.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,00:00:02,122.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,00:00:03,122.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,00:00:04,122.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
195,NaN,00:03:15,164.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
196,NaN,00:03:16,165.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
197,NaN,00:03:17,165.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
198,NaN,00:03:18,166.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
